# ELI Index Diagnostics — E3SM Hindcast (ne30pg2)

This notebook starts from the monthly ELI index files produced by
`0_run_eli_index_e3sm.ipynb`.

It computes drift-corrected ELI anomalies, evaluates ACC and RMSE skill
against the observed ERSSTv5-based ELI, and produces lead-time diagnostic
plots.

**Prerequisites:** run `0_run_eli_index_e3sm.ipynb` first to generate  
`OUTDIR/E3SMLE{mm}_ELI_N{nens}_M{nlead}.nc`.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import sys
import re
from pathlib import Path

# Resolve native-library paths from the active kernel.
_env_prefix = sys.prefix
_proj_path  = os.path.join(_env_prefix, "share", "proj")
if os.path.isfile(os.path.join(_proj_path, "proj.db")):
    os.environ["CONDA_PREFIX"] = _env_prefix
    os.environ["PROJ_LIB"]    = _proj_path
    os.environ["PROJ_DATA"]   = _proj_path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import pearsonr

from esp_lab import stats

print(f"Python      : {sys.executable}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', '(not set)')}")

## Configuration — fill in ALL fields before running

Fields left as `None` will cause the **Validate** cell to raise an error.

In [ ]:
from pathlib import Path
import numpy as np

# ------------------------------------------------------------------ #
#  PATHS
# ------------------------------------------------------------------ #

# Directory where 0_run_eli_index_e3sm.ipynb wrote its output.
ELI_DATA_DIR  = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/E3SMLE")

# Observational ELI (ERSSTv5), provided by L. Vroekel.
OBS_ELI_FILE  = Path(
    "/global/cfs/cdirs/e3sm/lvroekel/ELI_ERSSTv5_1854.01-2020.05.xlsx"
)

# Diagnostics output directory.
OUTDIR        = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/E3SMLE")

# Figure output directory.
FIGURE_OUTDIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)


def figure_filename(*parts, ext="png"):
    """Build consistent, readable lowercase snake_case figure filenames."""
    clean_parts = ["fig"]
    for part in parts:
        if part is None:
            continue
        text = re.sub(r"[^A-Za-z0-9]+", "_", str(part).strip()).strip("_").lower()
        if text:
            clean_parts.append(text)
    return "_".join(clean_parts) + f".{ext.lstrip('.')}"


# ------------------------------------------------------------------ #
#  HINDCAST EXPERIMENT
# ------------------------------------------------------------------ #

CASE_PREFIX = "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL"

START_YEAR  = 1980
END_YEAR    = 2018
EXCL_YEAR   = None

INIT_MONTHS = [5, 11]     # May and November starts

lead_years = [y for y in np.arange(START_YEAR, END_YEAR + 1) if y != EXCL_YEAR]

# ------------------------------------------------------------------ #
#  ENSEMBLE
# ------------------------------------------------------------------ #

CASE_NENS = 10
MEMBERS   = [f"EN{i:02d}" for i in range(CASE_NENS)]
NLEAD     = 24

# ------------------------------------------------------------------ #
#  CLIMATOLOGY WINDOW (for drift removal)
# ------------------------------------------------------------------ #

CLIM_Y0 = 1980
CLIM_Y1 = 2010

# ------------------------------------------------------------------ #
#  RUN CONTROL
# ------------------------------------------------------------------ #

OVERWRITE_OUTPUT = True

print(f"START_YEAR  : {START_YEAR}")
print(f"END_YEAR    : {END_YEAR}")
print(f"INIT_MONTHS : {INIT_MONTHS}")
print(f"CASE_NENS   : {CASE_NENS}")
print(f"NLEAD       : {NLEAD}")
print(f"CLIM window : {CLIM_Y0}–{CLIM_Y1}")

## Validate — run this before anything else

In [ ]:
_required = {
    "ELI_DATA_DIR":  ELI_DATA_DIR,
    "OBS_ELI_FILE":  OBS_ELI_FILE,
    "OUTDIR":        OUTDIR,
    "INIT_MONTHS":   INIT_MONTHS,
}
_missing = [k for k, v in _required.items() if v is None]
if _missing:
    raise ValueError(
        "Required fields are still None:\n" + "\n".join(f"  {k}" for k in _missing)
    )

_errors = []
if not ELI_DATA_DIR.is_dir():
    _errors.append(f"ELI_DATA_DIR not found: {ELI_DATA_DIR}")
if not OBS_ELI_FILE.is_file():
    _errors.append(f"OBS_ELI_FILE not found: {OBS_ELI_FILE}")

for m in INIT_MONTHS:
    _f = ELI_DATA_DIR / f"E3SMLE{m:02d}_ELI_N{CASE_NENS:02d}_M{NLEAD:02d}.nc"
    if not _f.is_file():
        _errors.append(f"ELI index file not found: {_f}")
if _errors:
    raise FileNotFoundError(
        "Path validation failed (run 0_run_eli_index_e3sm.ipynb first):\n"
        + "\n".join(_errors)
    )

OUTDIR.mkdir(parents=True, exist_ok=True)

print("Configuration valid ✓")
print(f"  ELI_DATA_DIR : {ELI_DATA_DIR}")
print(f"  OBS_ELI_FILE : {OBS_ELI_FILE}")
print(f"  OUTDIR       : {OUTDIR}")
for m in INIT_MONTHS:
    _f = ELI_DATA_DIR / f"E3SMLE{m:02d}_ELI_N{CASE_NENS:02d}_M{NLEAD:02d}.nc"
    print(f"  ELI file ({m:02d}) : {_f.name}  ✓")

## 1.  Load observational ELI

The Excel file contains a monthly ELI time series derived from ERSSTv5.
Each column header is a year; each row is a calendar month (0-indexed).
Values are converted to an `xarray.DataArray` with a monthly time axis.

In [ ]:
df_obs = pd.read_excel(OBS_ELI_FILE)

# Collect all available years in column order.
obs_years = [c for c in df_obs.columns if isinstance(c, (int, np.integer))]

times_obs    = []
eli_obs_vals = []
for yr in obs_years:
    for mo in range(12):
        val = df_obs[yr].iloc[mo]
        times_obs.append(pd.Timestamp(year=int(yr), month=mo + 1, day=1))
        eli_obs_vals.append(float(val) if pd.notna(val) else np.nan)

obs_time = pd.DatetimeIndex(times_obs)
eli_obs = xr.DataArray(
    np.array(eli_obs_vals, dtype=np.float32),
    dims="time",
    coords={"time": obs_time},
    name="eli_obs",
    attrs={
        "long_name": "Observed Equatorial Longitude Index (ERSSTv5)",
        "units": "degrees_east",
        "source": str(OBS_ELI_FILE),
    },
)

# Clip to model verification period (with buffer for drift removal).
eli_obs = eli_obs.sel(time=slice(str(START_YEAR - 2), str(END_YEAR + 2)))

# Monthly climatology over the base period for anomaly computation.
obs_clim_mask = (
    (eli_obs.time.dt.year >= CLIM_Y0) &
    (eli_obs.time.dt.year <= CLIM_Y1)
)
obs_clim = eli_obs.isel(time=obs_clim_mask).groupby("time.month").mean()

eli_obs_anom = eli_obs.groupby("time.month") - obs_clim
eli_obs_anom.name = "eli_obs_anom"
eli_obs_anom.attrs = {
    "long_name": "Observed ELI anomaly (ERSSTv5)",
    "units": "degrees_east",
}

print(f"Observational ELI: {len(eli_obs)} months")
print(f"  period : {str(eli_obs.time.values[0])[:10]} — {str(eli_obs.time.values[-1])[:10]}")
print(f"  min/max: {float(eli_obs.min()):.2f} / {float(eli_obs.max()):.2f}°E")
print(f"  mean   : {float(eli_obs.mean()):.2f}°E")
print(f"\nObs anomaly std : {float(eli_obs_anom.std()):.2f}°E")

## 2.  Load model ELI

In [ ]:
# Load one NetCDF per init month produced by 0_run_eli_index_e3sm.ipynb.
# Shape: (Y=init_year, L=lead_month, M=member).

eli_model = {}  # keyed by init_month
eli_time  = {}  # absolute calendar time for (Y, L) grid

for m in INIT_MONTHS:
    ncfile = ELI_DATA_DIR / f"E3SMLE{m:02d}_ELI_N{CASE_NENS:02d}_M{NLEAD:02d}.nc"
    ds = xr.open_dataset(ncfile).load()
    eli_model[m] = ds["eli"]

    # Build absolute target calendar: init year + lead offset (months).
    # Lead 1 = month after init; e.g. init Nov 1980 + lead 1 = Dec 1980.
    years_da = ds["Y"].values
    leads_da = ds["L"].values
    t_grid = np.array(
        [
            [
                pd.Timestamp(year=int(yr), month=m, day=1)
                + pd.DateOffset(months=int(lead))
                for lead in leads_da
            ]
            for yr in years_da
        ],
        dtype="datetime64[ns]",
    )
    eli_time[m] = xr.DataArray(
        t_grid,
        dims=("Y", "L"),
        coords={"Y": years_da, "L": leads_da},
        name="time",
    )

    ds.close()

    n_valid = int(np.isfinite(eli_model[m]).sum())
    n_total = int(eli_model[m].size)
    print(f"Init month {m:02d}: {eli_model[m].dims}  {eli_model[m].shape}  "
          f"({n_valid}/{n_total} valid values)")
    print(f"  range: {float(eli_model[m].min()):.2f} – {float(eli_model[m].max()):.2f}°E")

## 3.  Drift removal

For each init month, subtract the lead-dependent ELI climatology computed
over the `CLIM_Y0`–`CLIM_Y1` base period.  The result is an ELI *anomaly*
relative to the model's own drift, aligned with the observed anomaly for
skill comparisons.

In [ ]:
eli_dd    = {}   # drift-corrected model ELI anomaly  (Y, L, M)
eli_drift = {}   # lead-dependent model ELI climatology (L,)

for m in INIT_MONTHS:
    da = eli_model[m]               # (Y, L, M)
    years = da["Y"].values

    # Select climatology years.
    clim_mask = (years >= CLIM_Y0) & (years <= CLIM_Y1)
    da_clim = da.isel(Y=clim_mask)

    # Lead-dependent mean over ensemble members and clim years.
    drift = da_clim.mean(dim=("Y", "M"))   # (L,)
    anom  = da - drift                      # (Y, L, M)  — broadcast over Y and M

    eli_dd[m]    = anom
    eli_drift[m] = drift

    print(f"Init month {m:02d}:  drift range = "
          f"{float(drift.min()):.2f} – {float(drift.max()):.2f}°E  "
          f"| anom mean = {float(anom.mean()):.3f}°E")

EN00/archive/ocn/hist
EN01/archive/ocn/hist
EN02/archive/ocn/hist
EN03/archive/ocn/hist
EN04/archive/ocn/hist
EN05/archive/ocn/hist
EN06/archive/ocn/hist
EN07/archive/ocn/hist
EN08/archive/ocn/hist
EN09/archive/ocn/hist


## 4.  Skill computation

For each init month and each lead, compute ACC (anomaly correlation) and
normalised RMSE between the ensemble-mean ELI anomaly and the observed ELI
anomaly, verifying only over the years in `lead_years`.

In [ ]:
skill = {}   # keyed by init_month

for m in INIT_MONTHS:
    da_anom = eli_dd[m]              # (Y, L, M)
    t_grid  = eli_time[m]            # (Y, L) datetime64
    leads   = da_anom["L"].values

    acc   = np.full(len(leads), np.nan)
    pval  = np.full(len(leads), np.nan)
    rmse  = np.full(len(leads), np.nan)
    nrmse = np.full(len(leads), np.nan)

    for li, lead in enumerate(leads):
        ens_mean = da_anom.isel(L=li).mean(dim="M").values   # (Y,)
        t_lead   = pd.DatetimeIndex(t_grid.isel(L=li).values)

        # Align model ensemble mean with observed anomaly at each target month.
        obs_vals = np.array(
            [
                float(
                    eli_obs_anom.sel(time=t, method="nearest",
                                     tolerance=pd.Timedelta("31D"))
                ) if (eli_obs_anom.time.values[0] <= t.to_datetime64()
                      <= eli_obs_anom.time.values[-1])
                else np.nan
                for t in t_lead
            ],
            dtype=np.float32,
        )

        valid = np.isfinite(ens_mean) & np.isfinite(obs_vals)
        if valid.sum() < 5:
            continue

        r, p  = pearsonr(ens_mean[valid], obs_vals[valid])
        acc[li]   = r
        pval[li]  = p
        e         = ens_mean[valid] - obs_vals[valid]
        rmse[li]  = np.sqrt(np.mean(e ** 2))
        nrmse[li] = rmse[li] / (np.std(obs_vals[valid]) + 1e-12)

    skill[m] = xr.Dataset(
        {
            "acc":   xr.DataArray(acc,   dims="lead", coords={"lead": leads}),
            "pval":  xr.DataArray(pval,  dims="lead", coords={"lead": leads}),
            "rmse":  xr.DataArray(rmse,  dims="lead", coords={"lead": leads},
                                  attrs={"units": "degrees_east"}),
            "nrmse": xr.DataArray(nrmse, dims="lead", coords={"lead": leads}),
        }
    )

    print(f"Init month {m:02d}:")
    print(f"  ACC  lead 1 = {acc[0]:.3f},  lead 12 = {acc[11]:.3f},  lead 24 = {acc[-1]:.3f}")
    print(f"  RMSE lead 1 = {rmse[0]:.2f}°E, lead 12 = {rmse[11]:.2f}°E, lead 24 = {rmse[-1]:.2f}°E")

## 5.  ELI ensemble time series vs observations

For each init month, plot the ensemble mean and spread (thin grey lines) of
the drift-corrected ELI anomaly together with the observed ELI anomaly.

In [ ]:
for m in INIT_MONTHS:
    da_anom = eli_dd[m]          # (Y, L, M)
    t_grid  = eli_time[m]        # (Y, L) datetime
    leads   = da_anom["L"].values
    years   = da_anom["Y"].values

    # Show first and last init years as two example panels.
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

    for ax, yi in zip(axes, [0, -1]):
        yr_val   = int(years[yi])
        ens_vals = da_anom.isel(Y=yi).values    # (L, M)
        ens_mean = np.nanmean(ens_vals, axis=-1)
        t_abs    = pd.DatetimeIndex(t_grid.isel(Y=yi).values)

        obs_vals = np.array(
            [
                float(
                    eli_obs_anom.sel(time=t, method="nearest",
                                     tolerance=pd.Timedelta("31D"))
                ) if (eli_obs_anom.time.values[0] <= t.to_datetime64()
                      <= eli_obs_anom.time.values[-1])
                else np.nan
                for t in t_abs
            ]
        )

        ax.plot(leads, ens_vals, color="gray", linewidth=0.5, alpha=0.5)
        ax.plot(leads, ens_mean, color="tab:blue", linewidth=2,
                label="E3SM ens mean")
        ax.plot(leads, obs_vals, color="k", linewidth=2, linestyle="--",
                label="OBS (ERSSTv5)")
        ax.axhline(0, color="gray", linewidth=0.7, linestyle=":")
        ax.set_xlabel("Lead month", fontsize=12)
        ax.set_ylabel("ELI anomaly (°E)", fontsize=12)
        ax.set_title(f"Init month {m:02d}, year {yr_val}", fontsize=12)
        ax.legend(fontsize=9, loc="upper right")
        ax.xaxis.set_major_locator(mticker.MultipleLocator(4))

    fig.suptitle(f"E3SM ELI Anomaly vs Observations — init month {m:02d}",
                 fontsize=13, y=1.01)
    plt.tight_layout()

    figname = figure_filename("eli", "ts", f"init{m:02d}")
    fig.savefig(FIGURE_OUTDIR / figname, dpi=150, bbox_inches="tight")
    print(f"Saved: {FIGURE_OUTDIR / figname}")
    plt.show()

## 6.  ACC and RMSE skill vs lead time

In [ ]:
colors_by_month = {5: "tab:orange", 11: "tab:blue"}
labels_by_month = {5: "Init May",  11: "Init Nov"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax_acc, ax_rmse = axes

for m in INIT_MONTHS:
    sk    = skill[m]
    leads = sk["lead"].values
    acc   = sk["acc"].values
    pval  = sk["pval"].values
    rmse  = sk["rmse"].values
    color = colors_by_month[m]
    label = labels_by_month[m]

    ax_acc.plot(leads, acc, color=color, linewidth=2, label=label)
    sig = pval < 0.05
    ax_acc.scatter(leads[sig], acc[sig], color=color, s=30, zorder=5)

    ax_rmse.plot(leads, rmse, color=color, linewidth=2, label=label)

ax_acc.axhline(0, color="gray", linewidth=0.8, linestyle=":")
ax_acc.set_xlabel("Lead month", fontsize=12)
ax_acc.set_ylabel("ACC", fontsize=12)
ax_acc.set_title("ELI Anomaly Correlation", fontsize=13)
ax_acc.set_ylim(-1, 1)
ax_acc.xaxis.set_major_locator(mticker.MultipleLocator(4))
ax_acc.legend(fontsize=10)

ax_rmse.set_xlabel("Lead month", fontsize=12)
ax_rmse.set_ylabel("RMSE (°E)", fontsize=12)
ax_rmse.set_title("ELI RMSE", fontsize=13)
ax_rmse.xaxis.set_major_locator(mticker.MultipleLocator(4))
ax_rmse.legend(fontsize=10)

for ax in axes:
    for sp in ax.spines.values():
        sp.set_linewidth(1.2)
    ax.tick_params(labelsize=11)

fig.suptitle("E3SM ELI Skill vs Lead Time", fontsize=14, y=1.02)
plt.tight_layout()

figname = figure_filename("eli", "skill", "acc", "rmse")
fig.savefig(FIGURE_OUTDIR / figname, dpi=150, bbox_inches="tight")
print(f"Saved: {FIGURE_OUTDIR / figname}")
plt.show()

## 7.  Lead-dependent drift diagnostic

Plot the model ELI lead-dependent climatology (drift) for each init month
alongside the observed ELI monthly climatology to assess systematic biases.

In [ ]:
fig, axes = plt.subplots(
    1, len(INIT_MONTHS),
    figsize=(7 * len(INIT_MONTHS), 4.5),
    sharey=True,
)
if len(INIT_MONTHS) == 1:
    axes = [axes]

for ax, m in zip(axes, INIT_MONTHS):
    drift = eli_drift[m].values   # (L,)
    leads = eli_model[m]["L"].values

    # Observed climatological ELI at the target months for each lead.
    obs_clim_leads = [
        float(obs_clim.sel(month=((m - 1 + int(lead)) % 12) + 1))
        for lead in leads
    ]

    ax.plot(leads, drift,          color="tab:blue", linewidth=2,
            label="Model drift (clim)")
    ax.plot(leads, obs_clim_leads, color="k",        linewidth=2,
            linestyle="--", label="OBS clim")

    ax.set_xlabel("Lead month", fontsize=12)
    ax.set_ylabel("ELI (°E)", fontsize=12)
    ax.set_title(f"Init month {m:02d}", fontsize=12)
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(4))
    for sp in ax.spines.values():
        sp.set_linewidth(1.2)

fig.suptitle("E3SM ELI Lead-Dependent Climatology vs OBS",
             fontsize=13, y=1.02)
plt.tight_layout()

figname = figure_filename("eli", "drift", "clim")
fig.savefig(FIGURE_OUTDIR / figname, dpi=150, bbox_inches="tight")
print(f"Saved: {FIGURE_OUTDIR / figname}")
plt.show()